In [ ]:
# utils
import json
from transformers import AutoTokenizer
from tqdm import tqdm
import re
import numpy as np
from collections import defaultdict
from scipy import integrate
import scipy
import math

## Set Variables

In [ ]:
model_flops = {
    'meta-llama/Llama-3.2-3B-Instruct': 3000000000,
    'Qwen/Qwen2.5-3B-Instruct': 3000000000,
    'google/gemma-3-4b-it': 4000000000,
    'Qwen/Qwen2.5-7B-Instruct': 7000000000,
    'google/gemma-3-27b-it': 27000000000,
}

# dataset="gsm8k"
dataset = "math"
model = 'meta-llama/Llama-3.2-3B-Instruct'
# model = "Qwen/Qwen2.5-3B-Instruct"
# model = 'google/gemma-3-4b-it'
cp_threshold = 0.90
beta_threshold = 0.95

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)
if dataset == 'omnimath':
    with open(f"./logs/self_certainty/sc_16_{dataset}_2048_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]
else:
    with open(f"./logs/self_certainty/sc_16_{dataset}_1024_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

In [ ]:
# load calibration data

if dataset == 'omnimath':
    with open(f"./logs/self_certainty_samples/sample_128_{dataset}_2048_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        calibration_data = [json.loads(line) for line in f]
else:
    with open(f"./logs/self_certainty_samples/sample_128_{dataset}_1024_self_certainty_{model.replace('/','_')}_seed_0.jsonl", "r") as f:
        calibration_data = [json.loads(line) for line in f]

print("Calibration data size:", len(calibration_data))

In [ ]:
def apply_chat_template(dataset, question, response):
    if dataset == "gpqa_diamond" or dataset == "arcChallenge":
        prompt = f"{question}\n\nBased on the above, what is the single, most likely answer choice? Answer in the format \"The correct answer is (insert answer here)\"."
        chat_template = [{'role': 'user', 'content': prompt}, {"role": "assistant", "content": response}]
    else:
        chat_template = [{'role': 'user', 'content': question}, {"role": "assistant", "content": response}]

    return tokenizer.apply_chat_template(chat_template, tokenize=False, add_generation_prompt=False)

def extract_boxed_content(text):
    start = text.find(r"\boxed{")
    if start == -1:
        return None
    i = start + len(r"\boxed{")
    depth = 1
    content = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        content.append(c)
        i += 1
    return "".join(content)

def clean(value):
    if dataset == 'gsm8k':
        value = value.replace(',','')
        numbers = re.findall(r"\d+(?:\.\d+)?", value)
        if len(numbers) > 0:
            value = numbers[-1]
        else:
            value= None
    else:
        final_value = extract_boxed_content(value)
        if final_value is None:
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = re.findall(r"A|B|C|D|a|b|c|d", value)
                value = value[0].lower() if value else ''
            else:
                value = ""
        else:
            value = final_value.replace(' ','')
            if dataset == 'gpqa_diamond' or dataset == 'arcChallenge':
                value = value.lower()
    return value

In [ ]:
# calculate self certainty variants
for inst in tqdm(data):
    sample = inst['self_certainty_per_token']
    responseLen = len(tokenizer.encode(inst['response'], add_special_tokens=False))
    sliding_window_size = 128
    clusters = []
    for i in range(0, len(sample)):
        window = sample[i:i+sliding_window_size]
        if window:
            cluster_avg = sum(window) / len(window)
            clusters.append(cluster_avg)
    
    lowest_group_confidence = min(clusters)
    avg_group_confidence = sum(clusters) / len(clusters)
    Bottom10group_confidence = sum(sorted(clusters)[:max(1, len(clusters)//10)]) / max(1, len(clusters)//10)
    inst['lowest_group_confidence'] = lowest_group_confidence
    inst['avg_group_confidence'] = avg_group_confidence
    inst['Bottom10group_confidence'] = Bottom10group_confidence


# also for calibration data
for inst in calibration_data:
    sample = inst['self_certainty_per_token']
    responseLen = len(tokenizer.encode(inst['response'], add_special_tokens=False))
    sliding_window_size = 128
    clusters = []
    for i in range(0, len(sample)):
        window = sample[i:i+sliding_window_size]
        if window:
            cluster_avg = sum(window) / len(window)
            clusters.append(cluster_avg)
    lowest_group_confidence = min(clusters)
    avg_group_confidence = sum(clusters) / len(clusters)
    Bottom10group_confidence = sum(sorted(clusters)[:max(1, len(clusters)//10)]) / max(1, len(clusters)//10)
    inst['lowest_group_confidence'] = lowest_group_confidence
    inst['avg_group_confidence'] = avg_group_confidence
    inst['Bottom10group_confidence'] = Bottom10group_confidence

In [ ]:
# threshold
predefined_confidences = []
correct_confidences = []
wrong_confidences = []
for inst in calibration_data:
    question = inst['question']
    response = inst['response']
    internal_value = inst['lowest_group_confidence']
    predefined_confidences.append(internal_value)
    verdict = inst['verdict']
    
    if verdict:
        correct_confidences.append(internal_value)
    else:
        wrong_confidences.append(internal_value)

predefined_confidence_mean = np.mean(predefined_confidences)
predefined_confidence_std = np.std(predefined_confidences)

candidates = np.sort(np.asarray(predefined_confidences, float))
best_tau = None
best_lo = None
for tau in candidates:
    predefined_confidences = np.array(predefined_confidences)
    correct_confidences = np.array(correct_confidences)
    total = len(predefined_confidences[predefined_confidences >= tau])
    successes = len(correct_confidences[correct_confidences >= tau])
    lo = successes / total if total > 0 else 0
    # print(f"tau: {tau}\tlo: {lo} ({successes} / {total})")
    if lo >= cp_threshold:
        best_tau = tau
        best_lo = lo
        break
    else:
        if best_lo is None or lo > best_lo:
            best_tau = tau
            best_lo = lo
print("best_tau:", best_tau)
print("best_lower_bound:", best_lo)


predefined_threshold = max(best_tau, np.mean(correct_confidences))
print("predefined threshold:", predefined_threshold)

In [ ]:
# aggregate results by question
results = defaultdict(list)
error_cnt=0
for inst in data:
    question = inst['question']


    if dataset == 'gsm8k':
        pred = inst['pred'].replace(' ','').strip()
        verdict = clean(inst['pred']) == clean(inst['answer'])
    else:
        pred = clean(inst['response'])
        if pred == '':
            pred = clean(inst['pred'])
            if pred == '':
                pred = inst['pred'].replace(' ','').strip()
        
        if pred == '':
            error_cnt += 1
        
        verdict = inst['verdict']

    response = inst['response']
    if pred == '':
        error_cnt+=1
    
    internal_value = inst['Bottom10group_confidence']
    
    if question not in results:
        results[question] = []
    
    if dataset == 'gsm8k':
        results[question].append((clean(pred), internal_value, verdict, apply_chat_template(dataset, question, response)))
    else:
        results[question].append((pred, internal_value, verdict, apply_chat_template(dataset, question, response)))


# log_predefined_confidences mean and std
log_predefined_confidences = np.log(np.array(predefined_confidences))
log_predefined_confidence_mean = np.mean(log_predefined_confidences)
log_predefined_confidence_std = np.std(log_predefined_confidences)

# ASC results

In [ ]:
majority_voting_results = defaultdict(list)
for question in results:
    majority_voting_results[question] = []
    for i in range(16):
        answer_dict={}
        answer_correct={}
        i = i+1
        for inst in results[question][:i]:
            answer , verdict = inst[0], inst[2]
            if answer not in answer_dict:
                answer_dict[answer] = 0
            answer_dict[answer] += 1
            if answer not in answer_correct:
                answer_correct[answer] = {}
            answer_correct[answer][verdict] = answer_correct[answer].get(verdict, 0) + 1

        
        final_pred = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)[0][0]
        sorted_answer_dict = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)

        if len(sorted_answer_dict) == 1:
            a, b = sorted_answer_dict[0][1], 0
        else:
            a, b = sorted_answer_dict[0][1], sorted_answer_dict[1][1]
        a = float(a)
        b = float(b)
        
        prob = integrate.quad(lambda x : x**(a) * (1-x)**(b), 0.5, 1)[0] / integrate.quad(lambda x : x**(a) * (1-x)**(b), 0, 1)[0]
        if dataset == 'arcChallenge' or dataset == 'gpqa_diamond':
            majority_voting_results[question].append((final_pred.lower() == question_answer[question].lower(), prob))
        else:
            majority_voting_results[question].append((answer_correct[final_pred].get(True, 0) >= answer_correct[final_pred].get(False, 0), prob))

asc_results=[]
threshold = 0.95
sample_size_list=[]
cnt=0
total_length = 0
for question in majority_voting_results:
    flag=False
    for i, inst in enumerate(majority_voting_results[question]):
        correctness, prob = inst
        response = results[question][i][3]
        total_length += len(tokenizer.encode(response, add_special_tokens=False))
        if prob >= threshold:
            flag = True
            sample_size_list.append(i+1)
            cnt += (correctness == True)
            asc_results.append(correctness == True)
            break
    if not flag:
        sample_size_list.append(len(majority_voting_results[question]))
        cnt += (majority_voting_results[question][-1][0] == True)
        asc_results.append(majority_voting_results[question][-1][0] == True)

print(f"====================ASC ({threshold})====================")
print("ASC Results:")
print("Correct:", cnt)
print("Total:", len(majority_voting_results))
print("Accuracy: {:.2f}%".format(cnt/len(majority_voting_results) * 100))
print("Average Sample Size:", np.mean(sample_size_list))
print("Average Response Length per Question:", total_length/len(majority_voting_results))
print("Total Response Length:", total_length)
print("Flops: ", total_length * model_flops[model], "FLOPS")
print("Average TFLOPS:", (total_length * model_flops[model]) / (len(sample_size_list) * 1e12))
print("==========================================")

In [ ]:
asc_sample_size_list = sample_size_list.copy()

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.stats import beta

# # Parameters
# a, b = 2.33, 1

# # Create x values
# x = np.linspace(0, 1, 500)

# # Beta PDF
# pdf = beta.pdf(x, a, b)

# # Compute P(p > 0.5)
# prob = 1 - beta.cdf(0.5, a, b)
# print("P(p > 0.5) =", prob)

In [ ]:
# plt.figure(figsize=(7, 4))

# # Plot PDF
# plt.plot(x, pdf, label=f'Beta({a},{b}) PDF')

# # Shade region p > 0.5
# mask = x >= 0.5
# plt.fill_between(x[mask], pdf[mask], alpha=0.4, label='P(p > 0.5)')
# # add annotation for the probability
# plt.text(0.6, max(pdf)*0.5, f'P(p > 0.5) = {prob:.2f}', fontsize=10, color='black')

# # Vertical line at threshold
# plt.axvline(0.5, color='red', linestyle='--', label='p = 0.5')

# plt.title(f"Beta({a},{b}) Distribution")
# plt.xlabel("p")
# plt.ylabel("Density")
# plt.legend()
# plt.grid(alpha=0.3)
# plt.show()

# Ours
$\frac{S(y)}{C_{ref}}$

In [ ]:
correct_confidence_mean = np.mean(correct_confidences)
wrong_confidence_mean = np.mean(wrong_confidences)

In [ ]:
import numpy as np

correct_updates = []
wrong_updates = []
lambda_value=0.7

majority_voting_results = defaultdict(list)
for question in tqdm(results):
    majority_voting_results[question] = []
    for i in range(16):
        answer_dict={}
        answer_correct={}
        i = i+1
        for inst in results[question][:i]:
            answer , self_certainty , verdict = inst[0], inst[1], inst[2]
            if answer not in answer_dict:
                answer_dict[answer] = 0
            update_value = max(1, np.exp(lambda_value*(self_certainty - predefined_confidence_mean) / (predefined_confidence_std)))
            answer_dict[answer] += update_value
            
            if answer in answer_correct:
                # correct_updates.append(update_value)
                correct_updates.append(self_certainty)
            else:
                # wrong_updates.append(update_value)
                wrong_updates.append(self_certainty)

            if answer not in answer_correct:
                answer_correct[answer] = {}

            

            answer_correct[answer][verdict] = answer_correct[answer].get(verdict, 0) + 1
        # print(final_pred)
        final_pred = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)[0][0]

        # calculate prob_1 > prob_2
        sorted_answer_dict = sorted(answer_dict.items(), key=lambda x: x[1], reverse=True)

        if len(sorted_answer_dict) == 1:
            a, b = sorted_answer_dict[0][1], 0
        else:
            a, b = sorted_answer_dict[0][1], sorted_answer_dict[1][1]
        a = float(a)
        b = float(b)
        try:
            prob = integrate.quad(lambda x : x**(a) * (1-x)**(b), 0.5, 1)[0] / integrate.quad(lambda x : x**(a) * (1-x)**(b), 0, 1)[0]
        except:
            print("Integration error:", sorted_answer_dict, a, b)
        if dataset == 'arcChallenge' or dataset == 'gpqa_diamond':
            majority_voting_results[question].append((final_pred.lower() == question_answer[question].lower(), prob))
        else:
            majority_voting_results[question].append((answer_correct[final_pred].get(True, 0) >= answer_correct[final_pred].get(False, 0), prob))

In [ ]:
ours_results=[]
threshold = beta_threshold
sample_size_list=[]
cnt=0
single_cnt=0
pass_cnt=0
total_pass_cnt=0
question_result={}
total_length = 0
for question in majority_voting_results:
    flag=False
    for i, inst in enumerate(majority_voting_results[question]):
        if i == 0:
            if results[question][0][1] >= predefined_threshold:
                sample_size_list.append(1)
                response = results[question][i][3]
                total_length += len(tokenizer.encode(response, add_special_tokens=False))
                cnt += (majority_voting_results[question][0][0] == True)
                single_cnt += (majority_voting_results[question][0][0] == True)
                ours_results.append(majority_voting_results[question][0][0] == True)
                flag = True
                question_result[question] = (majority_voting_results[question][0][0] == True, 1)
                break
            else:
                correctness, prob = inst
                pass_cnt += (correctness == False)
                total_pass_cnt += 1
    
        correctness, prob = inst
        response = results[question][i][3]
        total_length += len(tokenizer.encode(response, add_special_tokens=False))

        if prob >= threshold:
            flag = True
            sample_size_list.append(i+1)
            cnt += (correctness == True)
            ours_results.append(correctness == True)
            question_result[question] = (correctness == True, 2)
            break
    if not flag:
        sample_size_list.append(len(majority_voting_results[question]))
        cnt += (majority_voting_results[question][-1][0] == True)
        ours_results.append(majority_voting_results[question][-1][0] == True)
        question_result[question] = (majority_voting_results[question][-1][0] == True, 2)


print(f"====================Ours ({threshold}, cp_threshold: {cp_threshold})====================")
print("Ours Results:")
print("Correct:", cnt)
print("Total:", len(sample_size_list))
print("Accuracy: {:.2f}%".format(cnt/len(sample_size_list) * 100))
print("Average Sample Size:", np.mean(sample_size_list))
print("Average Response Length per Question:", total_length/len(sample_size_list))
print("Total Response Length:", total_length)
print("Flops: ", total_length * model_flops[model], "FLOPS")
print("Average TFLOPS:", round((total_length * model_flops[model]) / (len(sample_size_list) * 1e12), 2))
print("Single Sample Correct Accuracy:", single_cnt, sample_size_list.count(1), single_cnt / sample_size_list.count(1) * 100 if sample_size_list.count(1) > 0 else 0)
print("==========================================")